# 🎨 Fashion Item Classification Pipeline
## Hierarchical Multi-Task Learning for Fashion Attributes

This notebook implements a comprehensive pipeline for classifying fashion items into hierarchical categories using attention-based multi-task learning with uncertainty weighting.

**Key Features:**
- ✅ FashionCLIP image embeddings (512-dim)
- ✅ Hierarchical multi-task classification (main → sub → category → related)
- ✅ Attention fusion layers for cross-task reasoning
- ✅ Uncertainty-weighted loss functions
- ✅ Per-class balanced sampling
- ✅ Threshold tuning and calibration
- ✅ Pseudo-labeling for unlabeled items
- ✅ Outfit compatibility dataset generation

## Section 1: Setup and Load Dataset

### ⚙️ Embedding Configuration

**This notebook is configured to always use 512-dimensional FashionCLIP embeddings** for all experiments and hyperparameter tuning.

- **FASHIONCLIP_EMBED_DIM = 512** (defined at notebook start)
- All validation checks ensure this dimension throughout
- All models, datasets, and experiments inherit this standard
- This guarantees reproducibility and consistent comparison across all hyperparameter configurations

**Key safeguards:**
- ✅ Embeddings validated at load time
- ✅ Dataset validates input dimension
- ✅ Model creation confirms num_embed_dim matches
- ✅ Hyperparameter tuning validates embeddings before each run


In [1]:
import json
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm

# ============================================================================
# 🔧 CONFIGURATION: Fashion Embedding Settings
# ============================================================================
# This ensures ALL experiments use consistent FashionCLIP embeddings
FASHIONCLIP_EMBED_DIM = 512  # FashionCLIP always outputs 512-dim embeddings
# ============================================================================

# Download latest version
import kagglehub

path = os.path.join(kagglehub.dataset_download("enisteper1/polyvore-outfit-dataset"), "polyvore_outfits")
print("✅ Dataset path:", path)

# Set up output directory for this notebook
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
output_dir = os.path.join(notebook_dir, "outputs_fashion_classification")
os.makedirs(output_dir, exist_ok=True)
print("✅ Output directory:", output_dir)

# Print configuration
print(f"✅ Embedding configuration: FASHIONCLIP_EMBED_DIM={FASHIONCLIP_EMBED_DIM}")

c:\TFM\APP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dataset path: C:\Users\gorka\.cache\kagglehub\datasets\enisteper1\polyvore-outfit-dataset\versions\2\polyvore_outfits
✅ Output directory: c:\TFM\APP\ml_pipeline\notebooks\outputs_fashion_classification
✅ Embedding configuration: FASHIONCLIP_EMBED_DIM=512


In [2]:
# Configure paths
metadata_path = os.path.join(path, 'polyvore_item_metadata.json')
images_path = os.path.join(path, 'images')
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load item metadata
with open(metadata_path, 'r') as f:
    item_metadata = json.load(f)

# Load category metadata
categories_df = pd.read_csv(os.path.join(path, 'categories.csv'))

print(f"✅ Loaded {len(item_metadata)} items")
print(f"✅ Loaded {len(categories_df)} category mappings")
print(f"✅ Device: {device}")
print(f"\nCategory distribution:")
print(categories_df['main_category'].value_counts())

✅ Loaded 251008 items
✅ Loaded 194 category mappings
✅ Device: cuda

Category distribution:
main_category
tops            35
shoes           32
bottoms         31
accessories     22
bags            18
outerwear       13
all-body        11
jewellery        9
hats             6
jewellery        4
accessories      4
scarves          2
sunglasses       2
Name: count, dtype: int64


## Section 2: Build Vocabularies and Prepare Data

In [3]:
def build_embedding_df(item_metadata, categories_df, min_related=250, min_category=50):
    """Build dataframe with multi-hot encoded labels"""
    # Clean category text
    categories_df = categories_df.copy()
    categories_df['main_category'] = categories_df['main_category'].str.strip().str.lower()
    categories_df['sub_category'] = categories_df['sub_category'].str.strip().str.lower()

    # Count tag frequencies
    related_counter = Counter()
    categories_counter = Counter()
    for item in item_metadata.values():
        related_counter.update(item.get('related', []))
        categories_counter.update(item.get('categories', []))

    # Filter tags by frequency
    filtered_related = [tag for tag, count in related_counter.items() if count >= min_related]
    filtered_categories = [cat for cat, count in categories_counter.items() if count >= min_category]

    print(f"✅ Related tags with ≥{min_related} appearances: {len(filtered_related)}")
    print(f"✅ Category tags with ≥{min_category} appearances: {len(filtered_categories)}")

    # Build vocabularies
    related_vocab = {tag: idx for idx, tag in enumerate(filtered_related)}
    category_vocab = {cat: idx for idx, cat in enumerate(filtered_categories)}
    main_vocab = {cat: idx for idx, cat in enumerate(categories_df['main_category'].dropna().unique())}
    sub_vocab = {cat: idx for idx, cat in enumerate(categories_df['sub_category'].dropna().unique())}

    # Build filtered DataFrame
    rows = []
    for item_id, item in item_metadata.items():
        related_ids = [related_vocab[tag] for tag in item.get('related', []) if tag in related_vocab]
        category_ids = [category_vocab[cat] for cat in item.get('categories', []) if cat in category_vocab]

        cat_id = item.get('category_id')
        cat_row = categories_df[categories_df['category_id'] == int(cat_id)] if cat_id else pd.DataFrame()

        main_ids = [main_vocab[m] for m in cat_row['main_category'].dropna().unique() if m in main_vocab]
        sub_ids = [sub_vocab[s] for s in cat_row['sub_category'].dropna().unique() if s in sub_vocab]

        rows.append({
            'item_id': item_id,
            'related_indices': related_ids,
            'category_indices': category_ids,
            'main_category_indices': main_ids,
            'sub_category_indices': sub_ids
        })

    return pd.DataFrame(rows), related_vocab, category_vocab, main_vocab, sub_vocab

# Build vocabularies
all_df, related_vocab, category_vocab, main_vocab, sub_vocab = build_embedding_df(item_metadata, categories_df)
print(f"\n✅ Created vocabulary-indexed dataframe with {len(all_df)} items")
print(all_df.head())

✅ Related tags with ≥250 appearances: 312
✅ Category tags with ≥50 appearances: 210

✅ Created vocabulary-indexed dataframe with 251008 items
     item_id related_indices category_indices main_category_indices  \
0  211990161              []               []                   [2]   
1  183179503              []        [0, 1, 2]                   [1]   
2  152771755              []               []                  [10]   
3  190445143       [0, 1, 2]     [0, 1, 2, 3]                   [1]   
4  211444470       [3, 4, 5]        [0, 1, 4]                   [1]   

  sub_category_indices  
0                  [8]  
1             [23, 24]  
2                 [59]  
3             [23, 24]  
4                 [25]  

✅ Created vocabulary-indexed dataframe with 251008 items
     item_id related_indices category_indices main_category_indices  \
0  211990161              []               []                   [2]   
1  183179503              []        [0, 1, 2]                   [1]   
2  1527717

## Section 3: Precompute FashionCLIP Embeddings

In [4]:
import math
from PIL import Image

def _load_img(p):
    with Image.open(p) as im:
        return im.convert("RGB")

def precompute_embeddings_simple(df, fclip, images_path, out_path="embeddings.npy", batch_size=256, force=False):
    """Precompute and cache FashionCLIP embeddings"""
    if os.path.exists(out_path) and not force:
        print(f"✅ Loading cached embeddings from {out_path}")
        return np.load(out_path)

    device_str = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🔧 Encoding images on {device_str}")

    n = len(df)
    n_batches = math.ceil(n / batch_size)
    parts = []

    bar = tqdm(total=n_batches, desc="Embedding images", unit="batch", leave=True)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        paths = [os.path.join(images_path, f"{df.iloc[i]['item_id']}.jpg") for i in range(start, end)]

        imgs = []
        for p in paths:
            try:
                imgs.append(_load_img(p))
            except Exception as e:
                pass

        if not imgs:
            bar.update(1)
            continue

        try:
            embeds = fclip.encode_images(imgs, batch_size=len(imgs), device=device_str)
        except TypeError:
            embeds = fclip.encode_images(imgs, batch_size=len(imgs))

        parts.append(np.asarray(embeds, dtype=np.float32))
        bar.update(1)

    bar.close()

    if parts:
        all_embeds = np.vstack(parts)
    else:
        all_embeds = np.empty((0, FASHIONCLIP_EMBED_DIM), dtype=np.float32)

    if all_embeds.shape[0] > n:
        all_embeds = all_embeds[:n]

    # Validate embeddings have correct dimension
    assert all_embeds.shape[1] == FASHIONCLIP_EMBED_DIM, \
        f"❌ Embedding dimension mismatch: got {all_embeds.shape[1]}, expected {FASHIONCLIP_EMBED_DIM}"

    # Ensure output directory exists
    out_dir = os.path.dirname(out_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    np.save(out_path, all_embeds)
    print(f"✅ Saved embeddings to {out_path} (shape: {all_embeds.shape})")
    print(f"✅ Embedding dimension verified: {all_embeds.shape[1]} == {FASHIONCLIP_EMBED_DIM}")
    return all_embeds

# Install and load FashionCLIP
try:
    from fashion_clip.fashion_clip import FashionCLIP
    fclip = FashionCLIP("fashion-clip")
    print("✅ FashionCLIP loaded")
except ImportError:
    print("⚠️ Installing fashion-clip...")
    os.system("pip install fashion-clip")
    from fashion_clip.fashion_clip import FashionCLIP
    fclip = FashionCLIP("fashion-clip")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ FashionCLIP loaded


In [5]:
# Compute embeddings (adjust batch_size based on memory)
embeddings_path = os.path.join(output_dir, "embeddings.npy")
embeddings = precompute_embeddings_simple(all_df, fclip, images_path, embeddings_path, batch_size=512, force=False)
print(f"✅ Embeddings shape: {embeddings.shape}")

✅ Loading cached embeddings from c:\TFM\APP\ml_pipeline\notebooks\outputs_fashion_classification\embeddings.npy
✅ Embeddings shape: (251008, 512)
✅ Embeddings shape: (251008, 512)


## Section 4: Define Hierarchical Multi-Task Neural Network

In [6]:
import torch.nn as nn
import torch.nn.functional as F

class AttentionFusion(nn.Module):
    """Attention fusion between hidden features and logits"""
    def __init__(self, hidden_dim, num_in, num_out, dropout=0.3):
        super().__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key   = nn.Linear(num_in, hidden_dim)
        self.value = nn.Linear(num_in, hidden_dim)
        self.fc    = nn.Linear(hidden_dim, num_out)
        self.dropout = nn.Dropout(dropout / 2)

    def forward(self, h, logits):
        q = self.query(h)
        k = self.key(logits)
        v = self.value(logits)
        
        attn_scores  = torch.matmul(q, k.T) / (q.size(-1) ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        
        fused = torch.matmul(attn_weights, v)
        fused = self.dropout(fused + h)
        return self.fc(fused)

class HierarchicalMultiTaskModel(nn.Module):
    """Multi-task hierarchical model: main → sub → category → related"""
    def __init__(self, embed_dim, num_related, num_categories, num_main, num_sub,
                 hidden_dim=512, dropout=0.3, use_attention=True):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.main_head = nn.Linear(hidden_dim, num_main)
        self.use_attention = use_attention
        
        self.sub_head = AttentionFusion(hidden_dim, num_main, num_sub, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_sub)
        )
        
        self.category_head_attn = AttentionFusion(hidden_dim, num_main + num_sub, num_categories, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main + num_sub, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_categories)
        )
        
        self.related_head_attn = AttentionFusion(hidden_dim, num_main + num_sub, num_related, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main + num_sub, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_related)
        )

        refine_dim = hidden_dim // 2
        self.category_refine = nn.Sequential(
            nn.Linear(num_categories, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_categories)
        )
        self.related_refine = nn.Sequential(
            nn.Linear(num_related, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_related)
        )

    def forward(self, x):
        h = self.shared(x)
        main_logits = self.main_head(h)

        if self.use_attention:
            sub_logits = self.sub_head(h, main_logits)
        else:
            sub_logits = self.sub_head(torch.cat([h, main_logits], dim=1))

        ms_logits = torch.cat([main_logits, sub_logits], dim=1)

        if self.use_attention:
            category_logits = self.category_head_attn(h, ms_logits)
        else:
            category_logits = self.category_head_attn(torch.cat([h, ms_logits], dim=1))
        category_logits = self.category_refine(category_logits)

        if self.use_attention:
            related_logits = self.related_head_attn(h, ms_logits)
        else:
            related_logits = self.related_head_attn(torch.cat([h, ms_logits], dim=1))
        related_logits = self.related_refine(related_logits)

        return main_logits, sub_logits, category_logits, related_logits

print("✅ Model architecture defined")

✅ Model architecture defined


In [7]:
# ============================================
# IMPROVED MODEL VARIANTS FOR COMPARISON
# ============================================

class HierarchicalMultiTaskModelV2(nn.Module):
    """V2: Deep shared encoder with residual connections"""
    def __init__(self, embed_dim, num_related, num_categories, num_main, num_sub,
                 hidden_dim=512, dropout=0.3, use_attention=True):
        super().__init__()

        # Deeper shared encoder with residual connections
        self.shared = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Additional shared layers (residual blocks)
        self.shared_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ) for _ in range(2)
        ])
        
        self.main_head = nn.Linear(hidden_dim, num_main)
        self.use_attention = use_attention
        
        self.sub_head = AttentionFusion(hidden_dim, num_main, num_sub, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_sub)
        )
        
        self.category_head_attn = AttentionFusion(hidden_dim, num_main + num_sub, num_categories, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main + num_sub, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_categories)
        )
        
        # Enhanced related head with extra capacity
        self.related_head_attn = AttentionFusion(hidden_dim, num_main + num_sub, num_related, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main + num_sub, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_related)
        )

        refine_dim = hidden_dim // 2
        self.category_refine = nn.Sequential(
            nn.Linear(num_categories, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_categories)
        )
        
        # More capacity for related refinement
        self.related_refine = nn.Sequential(
            nn.Linear(num_related, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_related)
        )

    def forward(self, x):
        h = self.shared(x)
        
        # Residual connections in shared layers
        for layer in self.shared_layers:
            h_new = layer(h)
            h = h + h_new  # Residual connection
        
        main_logits = self.main_head(h)

        if self.use_attention:
            sub_logits = self.sub_head(h, main_logits)
        else:
            sub_logits = self.sub_head(torch.cat([h, main_logits], dim=1))

        ms_logits = torch.cat([main_logits, sub_logits], dim=1)

        if self.use_attention:
            category_logits = self.category_head_attn(h, ms_logits)
        else:
            category_logits = self.category_head_attn(torch.cat([h, ms_logits], dim=1))
        category_logits = self.category_refine(category_logits)

        if self.use_attention:
            related_logits = self.related_head_attn(h, ms_logits)
        else:
            related_logits = self.related_head_attn(torch.cat([h, ms_logits], dim=1))
        related_logits = self.related_refine(related_logits)

        return main_logits, sub_logits, category_logits, related_logits

print("✅ V2 Model (deeper with residual connections) defined")

class HierarchicalMultiTaskModelV3(nn.Module):
    """V3: Task-specific refinement heads for related task"""
    def __init__(self, embed_dim, num_related, num_categories, num_main, num_sub,
                 hidden_dim=512, dropout=0.3, use_attention=True):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.main_head = nn.Linear(hidden_dim, num_main)
        self.use_attention = use_attention
        
        self.sub_head = AttentionFusion(hidden_dim, num_main, num_sub, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_sub)
        )
        
        self.category_head_attn = AttentionFusion(hidden_dim, num_main + num_sub, num_categories, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main + num_sub, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_categories)
        )
        
        self.related_head_attn = AttentionFusion(hidden_dim, num_main + num_sub, num_related, dropout) if use_attention else nn.Sequential(
            nn.Linear(hidden_dim + num_main + num_sub, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_related)
        )

        refine_dim = hidden_dim // 2
        self.category_refine = nn.Sequential(
            nn.Linear(num_categories, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_categories)
        )
        
        # Task-specific multi-stage refinement for related (harder task)
        self.related_stage1 = nn.Sequential(
            nn.Linear(num_related + num_main + num_sub, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.related_stage2 = nn.Sequential(
            nn.Linear(hidden_dim, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_related)
        )

    def forward(self, x):
        h = self.shared(x)
        main_logits = self.main_head(h)

        if self.use_attention:
            sub_logits = self.sub_head(h, main_logits)
        else:
            sub_logits = self.sub_head(torch.cat([h, main_logits], dim=1))

        ms_logits = torch.cat([main_logits, sub_logits], dim=1)

        if self.use_attention:
            category_logits = self.category_head_attn(h, ms_logits)
        else:
            category_logits = self.category_head_attn(torch.cat([h, ms_logits], dim=1))
        category_logits = self.category_refine(category_logits)

        if self.use_attention:
            related_logits = self.related_head_attn(h, ms_logits)
        else:
            related_logits = self.related_head_attn(torch.cat([h, ms_logits], dim=1))
        
        # Multi-stage refinement for related task using outputs from other heads
        related_combined = torch.cat([related_logits, main_logits, sub_logits], dim=1)
        related_refined = self.related_stage1(related_combined)
        related_logits = self.related_stage2(related_refined)

        return main_logits, sub_logits, category_logits, related_logits

print("✅ V3 Model (task-specific refinement for related) defined")

class HierarchicalMultiTaskModelV4(nn.Module):
    """V4: Multi-head attention within heads"""
    def __init__(self, embed_dim, num_related, num_categories, num_main, num_sub,
                 hidden_dim=512, dropout=0.3, use_attention=True, num_attn_heads=4):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.main_head = nn.Linear(hidden_dim, num_main)
        self.use_attention = use_attention
        
        # Multi-head attention for sub task
        self.sub_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim, 
            num_heads=num_attn_heads,
            dropout=dropout,
            batch_first=True
        )
        self.sub_linear = nn.Linear(hidden_dim + num_main, num_sub)
        
        # Multi-head attention for category task
        self.category_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_attn_heads,
            dropout=dropout,
            batch_first=True
        )
        self.category_linear = nn.Linear(hidden_dim + num_main + num_sub, num_categories)
        
        # Multi-head attention for related task
        self.related_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_attn_heads,
            dropout=dropout,
            batch_first=True
        )
        self.related_linear = nn.Linear(hidden_dim + num_main + num_sub, num_related)

        refine_dim = hidden_dim // 2
        self.category_refine = nn.Sequential(
            nn.Linear(num_categories, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_categories)
        )
        self.related_refine = nn.Sequential(
            nn.Linear(num_related, refine_dim),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(refine_dim, num_related)
        )

    def forward(self, x):
        h = self.shared(x)
        h_expanded = h.unsqueeze(1) if h.dim() == 2 else h  # Add sequence dimension if needed
        
        main_logits = self.main_head(h)

        # Sub head with multi-head attention
        sub_attn, _ = self.sub_attention(h_expanded, h_expanded, h_expanded)
        sub_attn = sub_attn.squeeze(1) if sub_attn.dim() == 3 else sub_attn
        sub_input = torch.cat([sub_attn, main_logits], dim=1)
        sub_logits = self.sub_linear(sub_input)

        ms_logits = torch.cat([main_logits, sub_logits], dim=1)

        # Category head with multi-head attention
        cat_attn, _ = self.category_attention(h_expanded, h_expanded, h_expanded)
        cat_attn = cat_attn.squeeze(1) if cat_attn.dim() == 3 else cat_attn
        cat_input = torch.cat([cat_attn, ms_logits], dim=1)
        category_logits = self.category_linear(cat_input)
        category_logits = self.category_refine(category_logits)

        # Related head with multi-head attention
        rel_attn, _ = self.related_attention(h_expanded, h_expanded, h_expanded)
        rel_attn = rel_attn.squeeze(1) if rel_attn.dim() == 3 else rel_attn
        rel_input = torch.cat([rel_attn, ms_logits], dim=1)
        related_logits = self.related_linear(rel_input)
        related_logits = self.related_refine(related_logits)

        return main_logits, sub_logits, category_logits, related_logits

print("✅ V4 Model (multi-head attention within heads) defined")


✅ V2 Model (deeper with residual connections) defined
✅ V3 Model (task-specific refinement for related) defined
✅ V4 Model (multi-head attention within heads) defined


## Section 5: Create Training Dataset and DataLoaders

In [8]:
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

class FashionEmbeddingDataset(Dataset):
    def __init__(self, embeddings, df, num_related, num_categories, num_main, num_sub):
        # Validate embeddings have correct FashionCLIP dimension (512-dim)
        if embeddings.shape[1] != FASHIONCLIP_EMBED_DIM:
            raise ValueError(
                f"❌ Embedding dimension mismatch!\n"
                f"   Expected: {FASHIONCLIP_EMBED_DIM}-dim (FashionCLIP standard)\n"
                f"   Got: {embeddings.shape[1]}-dim\n"
                f"   All experiments must use 512-dim FashionCLIP embeddings"
            )
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.df = df.reset_index(drop=True)
        self.num_related = num_related
        self.num_categories = num_categories
        self.num_main = num_main
        self.num_sub = num_sub

    def __len__(self):
        return len(self.df)

    def _make_multihot(self, indices_list, size):
        vec = torch.zeros(size, dtype=torch.float32)
        if indices_list is not None and len(indices_list) > 0:
            idx = torch.tensor(indices_list, dtype=torch.long)
            vec[idx] = 1.0
        return vec

    def __getitem__(self, idx):
        emb = self.embeddings[idx]
        row = self.df.iloc[idx]

        related    = self._make_multihot(row.get("related_indices", None), self.num_related)
        categories = self._make_multihot(row.get("category_indices", None), self.num_categories)
        main_c     = self._make_multihot(row.get("main_category_indices", None), self.num_main)
        sub_c      = self._make_multihot(row.get("sub_category_indices", None), self.num_sub)

        return emb, related, categories, main_c, sub_c

# Vocabulary stats - Use FASHIONCLIP_EMBED_DIM constant throughout
num_embed_dim = FASHIONCLIP_EMBED_DIM
assert num_embed_dim == 512, f"❌ Expected embed_dim=512 (FashionCLIP standard), got {num_embed_dim}"
num_related = len(related_vocab)
num_categories = len(category_vocab)
num_main = len(main_vocab)
num_sub = len(sub_vocab)

print(f"📊 Vocabulary sizes:")
print(f"  - Related tags: {num_related}")
print(f"  - Categories: {num_categories}")
print(f"  - Main categories: {num_main}")
print(f"  - Sub categories: {num_sub}")

# Create dataset with validated 512-dim embeddings
print(f"\n✅ Creating FashionEmbeddingDataset")
print(f"   - Embedding dimension: {FASHIONCLIP_EMBED_DIM}-dim (FashionCLIP standard)")
print(f"   - Data points: {len(embeddings)}")
all_df_indexed = all_df.reset_index(drop=True)
dataset = FashionEmbeddingDataset(
    embeddings=embeddings,
    df=all_df_indexed,
    num_related=num_related,
    num_categories=num_categories,
    num_main=num_main,
    num_sub=num_sub
)

# Split 80/10/10
n = len(dataset)
n_test = int(n * 0.10)
n_val  = int(n * 0.10)
n_train = n - n_val - n_test

generator = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=generator)

# Create DataLoaders
batch_size = 512
num_workers = 0 if torch.cuda.is_available() else 0
pin_memory = False

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)

print(f"\n✅ Dataset splits:")
print(f"  - Train: {len(train_loader.dataset)}")
print(f"  - Val:   {len(val_loader.dataset)}")
print(f"  - Test:  {len(test_loader.dataset)}")

📊 Vocabulary sizes:
  - Related tags: 312
  - Categories: 210
  - Main categories: 11
  - Sub categories: 141

✅ Creating FashionEmbeddingDataset
   - Embedding dimension: 512-dim (FashionCLIP standard)
   - Data points: 251008

✅ Dataset splits:
  - Train: 200808
  - Val:   25100
  - Test:  25100

✅ Dataset splits:
  - Train: 200808
  - Val:   25100
  - Test:  25100


## Section 6: Training with Smart Label-Based Loss Masking

The model uses an **intelligent loss function** that adapts based on what labels actually exist:

### 🎯 Loss Masking Strategy:

**Two levels of masking work together:**

#### Level 1: Class-Level Masking (per-index/class)
```
For each index that can be predicted:

If target=1 (index IS labeled for this item):
  → loss_weight = 1.0 ⭐⭐⭐
  → "HEAVY PENALTY if you predict wrong here!"
  → Model learns: "Predict correctly for what IS labeled"

If target=0 (index is NOT labeled for this item):
  → loss_weight = 0.1 ⭐
  → "Light penalty for predicting this when it's not labeled"
  → Model learns: "Don't stress about predicting unlabeled things"
```

#### Level 2: Sample-Level Masking (entire item)
```
For each item:

If item has NO labels at all (completely empty):
  → sample_weight = 0.01 (almost NO penalty)
  → "Don't penalize me, I have no training data!"

If item has at least ONE label:
  → sample_weight = 1.0 (normal penalty)
  → "Penalize me equally for mistakes"
```

### 💡 How They Work Together:

```
SCENARIO 1: Item HAS labels (e.g., related=[5, 23, 41])
────────────────────────────────────────────────────────
Sample weight = 1.0
For index 5 (labeled):      loss = BCE × 1.0 × 1.0 = FULL STRENGTH ⭐⭐⭐
For index 42 (not labeled): loss = BCE × 0.1 × 1.0 = WEAK ⭐

Result: Model learns to predict 5, doesn't panic about 42

SCENARIO 2: Item has NO labels at all (empty related=[])
──────────────────────────────────────────────────────
Sample weight = 0.01
For ANY index:              loss = BCE × weight × 0.01 = TINY ⭐

Result: Model doesn't get penalized for predicting anything
```

### ✅ Why This Works:

1. **Main (always labeled)**: Full penalty for mistakes ✓
2. **Related/Category (may be empty)**: 
   - If labeled → Full penalty for mistakes
   - If empty → Almost no penalty for any prediction
3. **Model learns**: "Predict what exists, don't stress about what doesn't"

In [9]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, hamming_loss, accuracy_score, precision_recall_curve

def bce_loss_with_weight(logits, targets, weight=1.0):
    """
    Weighted BCE loss with class imbalance handling AND empty instance masking.
    
    Strategy:
    1. CLASS IMBALANCE HANDLING:
       - Calculate frequency of positive labels per class
       - Weight underrepresented classes (rare labels) higher
       - Weight overrepresented classes (common labels) lower
    
    2. EMPTY INSTANCE MASKING:
       - Instances with NO labels get much lower per-class weight
       - Prevents penalizing predictions on items with no training data
    
    3. FALSE POSITIVE PENALTY:
       - Wrong predictions (target=0, pred=1) get 1.5x penalty
    
    Combined effect: Balance rare vs common labels WHILE being lenient on empty items
    """
    # Per-class loss (not reduced yet)
    loss_per_class = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    
    # Get predictions from logits (positive if logit > 0)
    predictions = (logits > 0).float()
    
    # Calculate class frequencies for imbalance handling
    batch_size = targets.size(0)
    num_classes = targets.size(1)
    
    # Frequency of positive labels per class (0 to 1)
    class_freq = targets.sum(dim=0) / batch_size
    class_freq = torch.clamp(class_freq, min=0.01)  # Avoid division by zero
    
    # Inverse frequency weighting: rare classes get higher weight
    # If a class is 10% positive (0.1 freq) → weight = 1/0.1 = 10
    # If a class is 50% positive (0.5 freq) → weight = 1/0.5 = 2
    inverse_freq_weight = 1.0 / class_freq
    inverse_freq_weight = inverse_freq_weight / inverse_freq_weight.mean()  # Normalize to avoid exploding losses
    
    # Create per-class weight matrix (batch_size, num_classes)
    class_weight = inverse_freq_weight.unsqueeze(0).expand(batch_size, -1)
    
    # EMPTY INSTANCE MASKING: Identify items with no labels
    num_labels_per_sample = targets.sum(dim=1)  # Count labels per sample
    is_empty = (num_labels_per_sample == 0).float()  # 1 if empty, 0 if has labels
    
    # For empty instances: reduce class weights significantly (0.1 instead of normal)
    # For non-empty instances: keep normal imbalance-adjusted weights
    empty_mask = is_empty.unsqueeze(1).expand(batch_size, num_classes)  # (batch_size, num_classes)
    class_weight = torch.where(
        empty_mask == 1.0,
        class_weight * 0.1,  # Very low weight for empty items
        class_weight          # Normal imbalance-adjusted weight
    )
    
    # Additional penalty for false positives (wrong predictions)
    wrong_prediction = (targets == 0.0) & (predictions == 1.0)
    false_positive_weight = torch.where(wrong_prediction, torch.full_like(targets, 1.5), torch.ones_like(targets))
    
    # Combine weights: class imbalance weight × false positive weight × empty instance masking
    combined_weight = class_weight * false_positive_weight
    
    # Apply combined weights
    loss_per_class = loss_per_class * combined_weight
    
    # Average per sample
    loss_per_sample = loss_per_class.mean(dim=1)
    
    # Apply sample-level weights (lower weight for empty instances overall)
    return (loss_per_sample * weight).mean()

def compute_sample_weights(targets, head_name, device='cpu'):
    """
    Compute per-sample weights to handle empty instances.
    
    Strategy:
    - Empty instances (no labels at all): weight = 0.1 (very low penalty - almost no data to learn)
    - Non-empty instances: weight = 1.0 (normal penalty - has labels to learn from)
    
    This ensures we don't heavily penalize samples that have no labels,
    while maintaining normal penalties for samples with actual data.
    """
    batch_size = targets.size(0)
    num_labels = targets.sum(dim=1)  # Count how many labels per sample
    
    # Empty samples get 0.1 weight (10% penalty)
    # Non-empty samples get 1.0 weight (full penalty)
    weights = torch.where(
        num_labels == 0,
        torch.full((batch_size,), 0.1, device=device),  # Low penalty for empty
        torch.ones(batch_size, device=device)            # Full penalty for non-empty
    )
    return weights

class LossWeights(torch.nn.Module):
    """Learnable uncertainty-based loss weighting for multi-task learning"""
    def __init__(self, heads=['related','category','main','sub']):
        super().__init__()
        self.log_vars = torch.nn.Parameter(torch.zeros(len(heads)))

    def forward(self, losses):
        weighted = []
        for i, l in enumerate(losses):
            log_var = torch.clamp(self.log_vars[i], -5, 5)
            precision = torch.exp(-log_var)
            weighted.append(precision * l + log_var)
        return sum(weighted)

def find_best_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2*precisions*recalls/(precisions+recalls+1e-8)
    return thresholds[f1_scores.argmax()] if len(thresholds) > 0 else 0.5

def compute_metrics_for_head(logits, targets, threshold, tune_thresholds=False):
    if logits.size == 0 or targets.size == 0:
        return {'n_valid': 0, 'auc_mean': float('nan'), 'f1_micro': float('nan'), 
                'f1_macro': float('nan'), 'f1_weighted': float('nan'), 
                'precision_micro': float('nan'), 'recall_micro': float('nan'),
                'hamming_loss': float('nan'), 'subset_acc': float('nan')}, threshold

    from scipy.special import expit
    probs = expit(np.clip(logits, -50, 50))
    t = targets.astype(int)

    if tune_thresholds:
        threshold = find_best_threshold(t.ravel(), probs.ravel())

    aucs = []
    for c in range(probs.shape[1]):
        y = t[:, c]
        if y.sum() > 0 and (y.size - y.sum()) > 0:
            try:
                aucs.append(roc_auc_score(y, probs[:, c]))
            except Exception:
                aucs.append(float('nan'))

    preds = (probs >= threshold).astype(int)
    prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(t, preds, average='micro', zero_division=0)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(t, preds, average='macro', zero_division=0)
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(t, preds, average='weighted', zero_division=0)

    h_loss = hamming_loss(t, preds)
    subset_acc = accuracy_score(t, preds)

    metrics = {
        'n_valid': int(len(t)),
        'auc_mean': float(np.nanmean(aucs)) if np.any(~np.isnan(aucs)) else float('nan'),
        'f1_micro': float(f1_micro),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'precision_micro': float(prec_micro),
        'recall_micro': float(rec_micro),
        'hamming_loss': float(h_loss),
        'subset_acc': float(subset_acc)
    }
    return metrics, threshold

print("✅ Training utilities loaded (balanced penalties with reduced penalty for empty instances)")

✅ Training utilities loaded (balanced penalties with reduced penalty for empty instances)


In [10]:
def validate_epoch(model, val_loader, device='cpu', threshold=0.5, tune_thresholds=False):
    model.eval()
    heads = ['related', 'category', 'main', 'sub']
    store = {h: {'logits': [], 'targets': []} for h in heads}
    total_val_loss, n_batches = 0.0, 0

    with torch.no_grad():
        for batch in val_loader:
            emb, rel, cat, main_c, sub_c = batch
            emb = emb.to(device)
            rel, cat, main_c, sub_c = rel.to(device), cat.to(device), main_c.to(device), sub_c.to(device)

            main_logits, sub_logits, cat_logits, rel_logits = model(emb)

            rel_w  = compute_sample_weights(rel, 'related', device)
            cat_w  = compute_sample_weights(cat, 'category', device)
            main_w = compute_sample_weights(main_c, 'main', device)
            sub_w  = compute_sample_weights(sub_c, 'sub', device)

            vloss = 0.0
            vloss += bce_loss_with_weight(rel_logits, rel, rel_w)
            vloss += bce_loss_with_weight(cat_logits, cat, cat_w)
            vloss += bce_loss_with_weight(main_logits, main_c, main_w)
            vloss += bce_loss_with_weight(sub_logits, sub_c, sub_w)

            total_val_loss += vloss.item()
            n_batches += 1

            store['related']['logits'].append(rel_logits.detach().cpu().numpy())
            store['related']['targets'].append(rel.detach().cpu().numpy())
            store['category']['logits'].append(cat_logits.detach().cpu().numpy())
            store['category']['targets'].append(cat.detach().cpu().numpy())
            store['main']['logits'].append(main_logits.detach().cpu().numpy())
            store['main']['targets'].append(main_c.detach().cpu().numpy())
            store['sub']['logits'].append(sub_logits.detach().cpu().numpy())
            store['sub']['targets'].append(sub_c.detach().cpu().numpy())


    avg_val_loss = total_val_loss / (n_batches if n_batches else 1)

    thresholds, metrics = {}, {}
    for h in heads:
        logits = np.concatenate(store[h]['logits'], axis=0) if store[h]['logits'] else np.empty((0,0))
        targets = np.concatenate(store[h]['targets'], axis=0) if store[h]['targets'] else np.empty((0,0))
        metrics[h], thresholds[h] = compute_metrics_for_head(logits, targets, threshold[h], tune_thresholds)

    return avg_val_loss, metrics, thresholds

print("✅ Validation function defined")

✅ Validation function defined


In [15]:
def train_model(model, train_loader, val_loader, optimizer, device='cpu', epochs=20,
                clip_grad=1.0, early_stop=5, ckpt_path='best.pt', threshold=0.5, loss_config=None):
    """Train model with uncertainty weighting, threshold tuning, and configurable loss strategy"""
    model = model.to(device)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    loss_weighter = LossWeights().to(device)
    
    # Use default loss config if not provided
    if loss_config is None:
        loss_config = LOSS_CONFIGURATIONS.get("weighted_standard", {})

    history = {'train_loss': [], 'val_loss': [], 'metrics': [], 'thresholds': []}
    best_val = float('inf')
    no_imp = 0
    best_thresholds = {h: threshold for h in ['related','category','main','sub']}

    for epoch in range(1, epochs+1):
        model.train()
        train_loss, nb = 0.0, 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False):
            emb, rel, cat, main_c, sub_c = batch
            emb = emb.to(device)
            rel, cat, main_c, sub_c = rel.to(device), cat.to(device), main_c.to(device), sub_c.to(device)

            optimizer.zero_grad()
            main_logits, sub_logits, cat_logits, rel_logits = model(emb)

            # Apply configurable loss function
            l_rel  = bce_loss_with_custom_weight(rel_logits, rel, loss_config=loss_config, device=device)
            l_cat  = bce_loss_with_custom_weight(cat_logits, cat, loss_config=loss_config, device=device)
            l_main = bce_loss_with_custom_weight(main_logits, main_c, loss_config=loss_config, device=device)
            l_sub  = bce_loss_with_custom_weight(sub_logits, sub_c, loss_config=loss_config, device=device)

            loss = loss_weighter([l_rel, l_cat, l_main, l_sub])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

            train_loss += loss.item()
            nb += 1

        avg_train = train_loss / (nb if nb else 1)

        # Validation
        tune = (epoch % 10 == 0)
        avg_val, val_metrics, tuned_thresholds = validate_epoch(
            model, val_loader, device=device, threshold=best_thresholds, tune_thresholds=tune
        )

        if tune and tuned_thresholds:
            best_thresholds = tuned_thresholds

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['metrics'].append(val_metrics)
        history['thresholds'].append(best_thresholds)

        scheduler.step(avg_val)

        print(f"Epoch {epoch}/{epochs} | TrainLoss={avg_train:.4f} | ValLoss={avg_val:.6f} | BestVal={best_val:.6f}")
        if tune:
            print(f"🔧 Tuned thresholds: {best_thresholds}")

        if avg_val < best_val - 1e-6:
            best_val = avg_val
            no_imp = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_imp += 1

        if no_imp >= early_stop:
            print(f"⏹️ Early stopping at epoch {epoch}")
            break

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

    return model, history

print("✅ Training function defined")


✅ Training function defined


In [ ]:
# Initialize model and optimizer
print(f"🔧 Model initialization:")
print(f"   - Embedding dimension: {num_embed_dim} (FASHIONCLIP_EMBED_DIM = {FASHIONCLIP_EMBED_DIM})")
print(f"   - Using num_embed_dim: {num_embed_dim}")

assert num_embed_dim == FASHIONCLIP_EMBED_DIM == 512, \
    f"❌ Embedding dimension must be 512, got {num_embed_dim}"

model = HierarchicalMultiTaskModel(
    embed_dim=num_embed_dim,
    num_related=num_related,
    num_categories=num_categories,
    num_main=num_main,
    num_sub=num_sub,
    use_attention=True
)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=1e-2)

print(f"✅ Model initialized on {device}")
print(f"✅ All layers configured for {num_embed_dim}-dim FashionCLIP embeddings")

In [ ]:
checkpt_path = os.path.join(output_dir, 'checkpoint.pt')
# Train model (adjust epochs as needed)
trained_model, history = train_model(
    model, train_loader, val_loader, optimizer,
    epochs=500,  # Reduce for quick testing
    device=device,
    early_stop=15,
    ckpt_path=checkpt_path
)
print("✅ Training complete")

## Section 7: Visualize Training Progress

In [ ]:
def plot_training_history(history, heads=('related','category','main','sub')):
    """Plot training curves and metrics"""
    epochs = np.arange(1, len(history.get('train_loss', [])) + 1)
    if len(epochs) == 0:
        print("No epochs found in history")
        return

    # Loss curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, history.get('train_loss', []), label='Train Loss', marker='o')
    ax1.plot(epochs, history.get('val_loss', []), label='Val Loss', marker='s')
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training vs Validation Loss")
    ax1.legend()
    ax1.grid(True)

    # F1-micro per head
    for h in heads:
        f1_vals = [m[h]['f1_micro'] for m in history.get('metrics', [])]
        ax2.plot(epochs, f1_vals, label=f"{h} F1-micro", marker='.')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("F1-micro")
    ax2.set_title("F1-micro scores per head")
    ax2.legend()
    ax2.grid(True)
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)

## Section 8: Model Demo - Test Sample Predictions

In [ ]:
# Get final thresholds
best_thresholds = history['thresholds'][-1] if history['thresholds'] else {h: 0.5 for h in ['related','category','main','sub']}

# Evaluate on test set
test_loss, test_metrics, best_thresholds = validate_epoch(
    trained_model, test_loader, threshold=best_thresholds, device=device, tune_thresholds=True
)

print(f"\n📊 Test Set Performance:")
print(f"Loss: {test_loss:.6f}\n")
for head, metrics in test_metrics.items():
    print(f"  {head.upper()}:")
    print(f"    - F1-micro: {metrics['f1_micro']:.4f}")
    print(f"    - F1-macro: {metrics['f1_macro']:.4f}")
    print(f"    - AUC-mean: {metrics['auc_mean']:.4f}")
    print(f"    - Precision: {metrics['precision_micro']:.4f}")
    print(f"    - Recall: {metrics['recall_micro']:.4f}")

In [ ]:
import random

def decode_multi_hot(vec, vocab):
    if vec is None:
        return []
    labels = []
    for j, flag in enumerate(vec):
        if int(flag):
            labels.append(vocab.get(int(j), f"#{j}") if vocab else int(j))
    return labels

def topk_with_scores(probs, k, vocab):
    k = max(1, min(k, probs.shape[0]))
    idxs = np.argsort(-probs)[:k]
    return [(vocab.get(int(j), f"#{j}") if vocab else int(j), float(probs[j])) for j in idxs]

# Build reverse vocabularies for decoding
vocab_rel = {v: k for k, v in related_vocab.items()}
vocab_cat = {v: k for k, v in category_vocab.items()}
vocab_main = {v: k for k, v in main_vocab.items()}
vocab_sub = {v: k for k, v in sub_vocab.items()}

print("✅ Vocabulary decoders ready")

In [ ]:
def show_random_test_predictions(model, test_loader, dataset, sample_count=3, top_k=5, seed=42):
    """Display random test samples with predictions"""
    device_str = model.parameters().__next__().device
    model.eval()

    df_pos = dataset.reset_index(drop=True)
    loader_ds = test_loader.dataset
    candidate_indices = list(loader_ds.indices) if hasattr(loader_ds, "indices") else list(range(len(df_pos)))

    if not candidate_indices:
        print("No test samples found")
        return

    random.seed(seed)
    chosen = random.sample(candidate_indices, min(sample_count, len(candidate_indices)))

    thr = best_thresholds

    with torch.no_grad():
        for idx in chosen:
            row = df_pos.iloc[int(idx)]
            ds = loader_ds.dataset if hasattr(loader_ds, "dataset") else loader_ds
            sample = ds[int(idx)]
            
            emb = sample[0].to(device_str)
            if emb.dim() == 1:
                emb = emb.unsqueeze(0)

            main_logits, sub_logits, cat_logits, rel_logits = model(emb)

            rel_probs = torch.sigmoid(rel_logits).cpu().numpy()[0]
            cat_probs = torch.sigmoid(cat_logits).cpu().numpy()[0]
            main_probs = torch.sigmoid(main_logits).cpu().numpy()[0]
            sub_probs = torch.sigmoid(sub_logits).cpu().numpy()[0]

            # Ground truth
            rel_gt = sample[1].cpu().numpy() if sample[1] is not None else None
            cat_gt = sample[2].cpu().numpy() if sample[2] is not None else None
            main_gt = sample[3].cpu().numpy() if sample[3] is not None else None
            sub_gt = sample[4].cpu().numpy() if sample[4] is not None else None

            print(f"\n{'='*60}")
            print(f"Sample #{idx}")
            print(f"{'='*60}")
            
            print(f"\n🏷️  GROUND TRUTH:")
            print(f"  Main:     {decode_multi_hot(main_gt, vocab_main)}")
            print(f"  Sub:      {decode_multi_hot(sub_gt, vocab_sub)}")
            print(f"  Category: {decode_multi_hot(cat_gt, vocab_cat)}")
            print(f"  Related:  {decode_multi_hot(rel_gt, vocab_rel)}")

            print(f"\n🔮 TOP-{top_k} PREDICTIONS:")
            print(f"  Main:     {topk_with_scores(main_probs, top_k, vocab_main)}")
            print(f"  Sub:      {topk_with_scores(sub_probs, top_k, vocab_sub)}")
            print(f"  Category: {topk_with_scores(cat_probs, top_k, vocab_cat)}")
            print(f"  Related:  {topk_with_scores(rel_probs, top_k, vocab_rel)}")

            # Thresholded predictions
            rel_bin = (rel_probs >= thr['related']).astype(int)
            cat_bin = (cat_probs >= thr['category']).astype(int)
            main_bin = (main_probs >= thr['main']).astype(int)
            sub_bin = (sub_probs >= thr['sub']).astype(int)

            print(f"\n✅ THRESHOLDED PREDICTIONS (threshold={thr}):")
            print(f"  Main:     {decode_multi_hot(main_bin, vocab_main)}")
            print(f"  Sub:      {decode_multi_hot(sub_bin, vocab_sub)}")
            print(f"  Category: {decode_multi_hot(cat_bin, vocab_cat)}")
            print(f"  Related:  {decode_multi_hot(rel_bin, vocab_rel)}")

# Show random predictions
show_random_test_predictions(trained_model, test_loader, all_df_indexed, sample_count=5)

In [ ]:
# Save the trained model as an experiment
print("\n📁 Saving trained model as experiment...")

# Get model parameters for naming convention
hidden_dim = model.fc_shared.out_features if hasattr(model, 'fc_shared') else 512
dropout = model.dropout_rate if hasattr(model, 'dropout_rate') else 0.2
lr = 5e-4  # default learning rate used in training
optimizer_name = "sgd"  # SGD optimizer being used

# Create experiment folder with naming convention: {optimizer}_hd{hidden}_do{dropout:.2f}_lr{lr:.0e}_bs{batch_size}
experiment_name = f"{optimizer_name}_hd{hidden_dim}_do{dropout:.2f}_lr{lr:.0e}_bs{batch_size}_2"
experiment_folder = os.path.join(output_dir, experiment_name)
os.makedirs(experiment_folder, exist_ok=True)

# 1. Save model weights
torch.save(trained_model.state_dict(), os.path.join(experiment_folder, "model.pt"))
print(f"  ✅ Saved model weights")

# 2. Save history.json (training curves and all epoch metrics)
with open(os.path.join(experiment_folder, "history.json"), 'w') as f:
    json.dump(history, f, indent=2, default=str)
print(f"  ✅ Saved training history with {len(history.get('train_loss', []))} epochs")

# 3. Save summary.json (config + performance + all test metrics)
summary_data = {
    'model_type': 'trained_model',
    'config': {
        'hidden_dim': model.fc_shared.out_features if hasattr(model, 'fc_shared') else 'unknown',
        'use_attention': True,
        'epochs_trained': len(history.get('train_loss', [])),
    },
    'test_performance': {
        'test_loss': float(test_loss),
    },
    'test_metrics_per_head': test_metrics,
    'best_thresholds': best_thresholds,
}
with open(os.path.join(experiment_folder, "summary.json"), 'w') as f:
    json.dump(summary_data, f, indent=2, default=str)
print(f"  ✅ Saved summary with test metrics")

# 4. Save metrics.json (comprehensive detailed metrics for all heads)
metrics_detailed = {
    'test_loss': float(test_loss),
    'per_head_metrics': test_metrics
}
with open(os.path.join(experiment_folder, "metrics.json"), 'w') as f:
    json.dump(metrics_detailed, f, indent=2, default=str)
print(f"  ✅ Saved detailed metrics for {len(test_metrics)} heads")

# 5. Save scores.npy (loss value)
np.save(os.path.join(experiment_folder, "scores.npy"), 
        np.array([test_loss]))
print(f"  ✅ Saved scores")

print(f"\n✅ Trained model saved as experiment: {experiment_folder}")
print(f"   📊 Test Loss: {test_loss:.6f}")
for head_name, head_metrics in test_metrics.items():
    print(f"   📈 {head_name.upper()}: F1={head_metrics.get('f1_macro', 0):.4f}, AUC={head_metrics.get('auc_mean', 0):.4f}")



## Section 9: Generate Pseudo-Labels for All Items

In [ ]:
def predict_all_items(model, df, embeddings, device='cpu', threshold=0.5, threshold_multiplier=1.0):
    """
    Generate predictions for all items with adjustable thresholds.
    
    Parameters:
    -----------
    model : torch.nn.Module - The trained classification model
    df : pd.DataFrame - Input dataframe
    embeddings : np.ndarray - Pre-computed embeddings for all items
    device : str - Device to use ('cpu' or 'cuda')
    threshold : float or dict - Threshold(s) for predictions
    threshold_multiplier : float - Multiplier to adjust thresholds (< 1.0 = more predictions)
                                   Examples:
                                   - 1.0: Use original thresholds (default)
                                   - 0.7: Lower thresholds by 30% → predict MORE items
                                   - 0.5: Lower thresholds by 50% → predict MANY more items
                                   - 0.3: Lower thresholds by 70% → predict LOTS of items
    """
    model.eval()
    model = model.to(device)

    # Parse thresholds and apply multiplier
    if isinstance(threshold, dict):
        thr_rel = threshold.get('related', 0.5) * threshold_multiplier
        thr_cat = threshold.get('category', 0.5) * threshold_multiplier
        thr_main = threshold.get('main', 0.5) * threshold_multiplier
        thr_sub = threshold.get('sub', 0.5) * threshold_multiplier
    else:
        thr_rel = thr_cat = thr_main = thr_sub = threshold * threshold_multiplier

    print(f"\n🎯 Prediction settings:")
    print(f"   Threshold multiplier: {threshold_multiplier:.2f}x")
    print(f"   Related threshold: {thr_rel:.4f}")
    print(f"   Category threshold: {thr_cat:.4f}")
    print(f"   Main threshold: {thr_main:.4f}")
    print(f"   Sub threshold: {thr_sub:.4f}")

    preds_related = []
    preds_category = []
    preds_main = []
    preds_sub = []
    
    pred_counts = {'related': 0, 'category': 0, 'main': 0, 'sub': 0}

    with torch.no_grad():
        for i in tqdm(range(len(df)), desc="Predicting all items", leave=False):
            emb = torch.tensor(embeddings[i], dtype=torch.float32).to(device)
            if emb.dim() == 1:
                emb = emb.unsqueeze(0)

            main_logits, sub_logits, cat_logits, rel_logits = model(emb)

            rel_probs = torch.sigmoid(rel_logits).cpu().numpy()[0]
            cat_probs = torch.sigmoid(cat_logits).cpu().numpy()[0]
            main_probs = torch.sigmoid(main_logits).cpu().numpy()[0]
            sub_probs = torch.sigmoid(sub_logits).cpu().numpy()[0]

            rel_pred_indices = [j for j, p in enumerate(rel_probs) if p >= thr_rel]
            cat_pred_indices = [j for j, p in enumerate(cat_probs) if p >= thr_cat]
            main_pred_indices = [j for j, p in enumerate(main_probs) if p >= thr_main]
            sub_pred_indices = [j for j, p in enumerate(sub_probs) if p >= thr_sub]

            preds_related.append(rel_pred_indices)
            preds_category.append(cat_pred_indices)
            preds_main.append(main_pred_indices)
            preds_sub.append(sub_pred_indices)
            
            pred_counts['related'] += len(rel_pred_indices)
            pred_counts['category'] += len(cat_pred_indices)
            pred_counts['main'] += len(main_pred_indices)
            pred_counts['sub'] += len(sub_pred_indices)

    df_pred = all_df.copy()
    df_pred["related_indices_pred"] = preds_related
    df_pred["category_indices_pred"] = preds_category
    df_pred["main_indices_pred"] = preds_main
    df_pred["sub_indices_pred"] = preds_sub
    
    # Print statistics
    print(f"\n📊 Prediction Statistics:")
    print(f"   Total items: {len(df_pred)}")
    print(f"   Total related predictions: {pred_counts['related']} (avg per item: {pred_counts['related']/len(df_pred):.2f})")
    print(f"   Total category predictions: {pred_counts['category']} (avg per item: {pred_counts['category']/len(df_pred):.2f})")
    print(f"   Total main predictions: {pred_counts['main']} (avg per item: {pred_counts['main']/len(df_pred):.2f})")
    print(f"   Total sub predictions: {pred_counts['sub']} (avg per item: {pred_counts['sub']/len(df_pred):.2f})")
    
    return df_pred

# Generate pseudo-labels with adjustable prediction volume
# Use threshold_multiplier to control how many predictions to make:
# - 1.0 = original thresholds (fewer predictions)
# - 0.7 = 30% lower thresholds (more predictions)
# - 0.5 = 50% lower thresholds (many more predictions)
# - 0.3 = 70% lower thresholds (lots of predictions)

print("🔄 Generating predictions with different threshold multipliers...\n")

# Try different multipliers to see how many predictions we get
for multiplier in [1.0, 0.7, 0.5]:
    df_predictions = predict_all_items(
        trained_model, 
        all_df_indexed, 
        embeddings, 
        device=device, 
        threshold=best_thresholds,
        threshold_multiplier=multiplier
    )
    print()

# Choose which multiplier to use (comment/uncomment as needed)
# Using 0.7 for a good balance between quantity and quality
CHOSEN_MULTIPLIER = 0.7
df_with_predictions = predict_all_items(
    trained_model, 
    all_df_indexed, 
    embeddings, 
    device=device, 
    threshold=best_thresholds,
    threshold_multiplier=CHOSEN_MULTIPLIER
)
print(f"\n✅ Using multiplier {CHOSEN_MULTIPLIER} for final predictions")
print(f"✅ Generated predictions for {len(df_with_predictions)} items")
print(df_with_predictions[['related_indices_pred', 'category_indices_pred', 'main_indices_pred', 'sub_indices_pred']].head())


In [11]:

def evaluate_model(model, test_loader, device='cpu', threshold=None):
    """
    Evaluate model on test set and return metrics.
    
    Parameters:
    -----------
    model : torch.nn.Module - The trained model
    test_loader : DataLoader - Test data loader
    device : str - Device to use ('cpu' or 'cuda')
    threshold : dict - Thresholds for each head (optional, uses 0.5 if not provided)
    
    Returns:
    --------
    dict - Test metrics for each head with keys: f1_macro, f1_micro, auc_mean, precision_micro, recall_micro, etc.
    """
    if threshold is None:
        threshold = {'related': 0.5, 'category': 0.5, 'main': 0.5, 'sub': 0.5}
    
    # Use validate_epoch with tune_thresholds=False to just evaluate without tuning
    test_loss, test_metrics, _ = validate_epoch(
        model, test_loader, device=device, threshold=threshold, tune_thresholds=False
    )
    
    return test_metrics

print("✅ evaluate_model function defined")


✅ evaluate_model function defined


In [19]:
# ============================================
# Loss Configuration Grid - Try Different Loss Weighting Strategies
# ============================================

# Define different loss configurations to compare
LOSS_CONFIGURATIONS = {
    # 1. Current: Weighted loss with class imbalance handling (baseline)
    "weighted_standard": {
        "name": "Standard Weighted (current)",
        "description": "Class imbalance weight + false positive weight + empty instance masking",
        "use_class_weight": True,
        "use_false_positive_weight": True,
        "fp_weight_value": 1.5,  # Weight for false positives
        "use_empty_instance_masking": True,
        "empty_instance_weight": 0.1,
        "reduction": "mean"
    },
    
    # 2. Unweighted: Simple BCE loss without any weighting
    "unweighted": {
        "name": "Unweighted (no weights)",
        "description": "Simple BCE loss without any special weighting",
        "use_class_weight": False,
        "use_false_positive_weight": False,
        "use_empty_instance_masking": False,
        "reduction": "mean"
    },
    
    # 3. Light weighting: Reduced weight values
    "weighted_light": {
        "name": "Light Weighted (reduced)",
        "description": "Reduced class imbalance and false positive weights",
        "use_class_weight": True,
        "use_false_positive_weight": True,
        "fp_weight_value": 1.2,  # Lower than standard
        "use_empty_instance_masking": True,
        "empty_instance_weight": 0.2,  # Higher than standard (less masking)
        "reduction": "mean"
    },
    
    # 4. Class weight only: Only handle class imbalance, no false positive weighting
    "weighted_class_only": {
        "name": "Class Weight Only",
        "description": "Only class imbalance weighting, no false positive or empty instance handling",
        "use_class_weight": True,
        "use_false_positive_weight": False,
        "use_empty_instance_masking": False,
        "reduction": "mean"
    },
    
    # 5. Heavy weighting: Increased weight values
    "weighted_heavy": {
        "name": "Heavy Weighted (increased)",
        "description": "Increased class imbalance and false positive weights",
        "use_class_weight": True,
        "use_false_positive_weight": True,
        "fp_weight_value": 2.0,  # Higher than standard
        "use_empty_instance_masking": True,
        "empty_instance_weight": 0.05,  # Lower than standard (more masking)
        "reduction": "mean"
    },
    
    # 6. Empty instance masking only
    "weighted_empty_mask_only": {
        "name": "Empty Mask Only",
        "description": "Only empty instance masking, no class imbalance or false positive weighting",
        "use_class_weight": False,
        "use_false_positive_weight": False,
        "use_empty_instance_masking": True,
        "empty_instance_weight": 0.1,
        "reduction": "mean"
    }
}

print("="*120)
print("LOSS CONFIGURATIONS - Compare Different Weighting Strategies")
print("="*120)
for loss_name, loss_config in LOSS_CONFIGURATIONS.items():
    print(f"\n📊 {loss_name.upper()}")
    print(f"   Name: {loss_config['name']}")
    print(f"   Description: {loss_config['description']}")
    print(f"   Class Weight: {loss_config['use_class_weight']}")
    print(f"   FP Weight: {loss_config['use_false_positive_weight']} (value: {loss_config.get('fp_weight_value', 'N/A')})")
    print(f"   Empty Masking: {loss_config['use_empty_instance_masking']} (weight: {loss_config.get('empty_instance_weight', 'N/A')})")

# ============================================
# Configuration Grid - Multiple Model Variants
# ============================================

# Define all configurations to try with different model architectures
configurations = {
    # Baseline - Current best model
    "baseline_v1": {
        "model_variant": "V1",
        "hidden_dim": 512,
        "dropout": 0.20,
        "optimizer": "adamw",
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
        "loss_config": "weighted_standard",
        "description": "V1: Original model - baseline"
    },
    
    # V2: Deeper with residual connections
    "deeper_v2_small": {
        "model_variant": "V2",
        "hidden_dim": 512,
        "dropout": 0.25,
        "optimizer": "adamw",
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
        "loss_config": "weighted_standard",
        "description": "V2: Deeper encoder + residual + enhanced related head"
    },
    
    "deeper_v2_large": {
        "model_variant": "V2",
        "hidden_dim": 768,
        "dropout": 0.30,
        "optimizer": "adamw",
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
        "loss_config": "weighted_light",
        "description": "V2: Deeper + larger hidden dim with light weighting"
    },
    
    # V3: Task-specific refinement (focuses on related task)
    "taskspec_v3_small": {
        "model_variant": "V3",
        "hidden_dim": 512,
        "dropout": 0.20,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "loss_config": "weighted_standard",
        "description": "V3: Multi-stage related refinement + cross-head attention"
    },
    
    "taskspec_v3_large": {
        "model_variant": "V3",
        "hidden_dim": 768,
        "dropout": 0.25,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 5e-5,
        "loss_config": "weighted_heavy",
        "description": "V3: Larger model with task-specific refinement - heavy weights"
    },
    
    # V4: Multi-head attention
    "multihead_v4_small": {
        "model_variant": "V4",
        "hidden_dim": 512,
        "dropout": 0.25,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "loss_config": "unweighted",
        "description": "V4: Multi-head attention - try unweighted loss"
    },
    
    "multihead_v4_large": {
        "model_variant": "V4",
        "hidden_dim": 768,
        "dropout": 0.30,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 5e-5,
        "loss_config": "weighted_class_only",
        "description": "V4: Larger multi-head with class weight only"
    },
    
    # V4: High learning rate (faster convergence)
    "multihead_v4_fastlr": {
        "model_variant": "V4",
        "hidden_dim": 512,
        "dropout": 0.25,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "loss_config": "weighted_standard",
        "description": "V4: Multi-head with higher learning rate"
    },
    
    # Additional configurations to test different loss strategies with V1
    "v1_unweighted": {
        "model_variant": "V1",
        "hidden_dim": 512,
        "dropout": 0.20,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "loss_config": "unweighted",
        "description": "V1: Baseline with unweighted loss"
    },
    
    "v1_light_weight": {
        "model_variant": "V1",
        "hidden_dim": 512,
        "dropout": 0.20,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "loss_config": "weighted_light",
        "description": "V1: Baseline with light weighting"
    },
    
    "v1_heavy_weight": {
        "model_variant": "V1",
        "hidden_dim": 512,
        "dropout": 0.20,
        "optimizer": "adamw",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "loss_config": "weighted_heavy",
        "description": "V1: Baseline with heavy weighting"
    },
}

# Display configuration summary
print("="*120)
print("CONFIGURATION GRID - Multiple Model Architectures and Loss Strategies")
print("="*120)
print(f"\nTotal configurations to try: {len(configurations)}\n")

for config_name, config in configurations.items():
    print(f"📋 {config_name.upper()}")
    print(f"   Variant: {config['model_variant']} | Hidden: {config['hidden_dim']} | Dropout: {config['dropout']}")
    print(f"   Learning Rate: {config['learning_rate']} | Weight Decay: {config['weight_decay']}")
    print(f"   Loss Config: {config['loss_config']}")
    print(f"   {config['description']}\n")

# Generate experiment names from configurations
def generate_experiment_name(config_name: str, config: dict) -> str:
    """Generate experiment folder name based on configuration."""
    variant = config["model_variant"]
    hidden_dim = config["hidden_dim"]
    dropout = int(config["dropout"] * 100)
    optimizer = config["optimizer"]
    lr = f"{config['learning_rate']:.0e}".replace("e-0", "e-")
    wd = f"{config['weight_decay']:.0e}".replace("e-0", "e-")
    loss_cfg = config.get("loss_config", "std")[:3].lower()  # First 3 chars of loss config
    
    exp_name = f"v{variant}_hd{hidden_dim}_dr{dropout}_{optimizer}_lr{lr}_wd{wd}_loss{loss_cfg}"
    return exp_name

print("="*120)
print("GENERATED EXPERIMENT NAMES")
print("="*120)
for config_name, config in configurations.items():
    exp_name = generate_experiment_name(config_name, config)
    print(f"{config_name:25} → {exp_name}")

print("\n✅ Ready to train multiple model variants with different loss configurations!")
print("📊 Compare: V1 (baseline) vs V2 (residual) vs V3 (task-specific) vs V4 (multi-head)")
print("💡 Loss Strategies: Weighted (standard/light/heavy) vs Unweighted vs Class-Only")
print("🎯 Goal: Improve Related task from 37% to 50%+")


LOSS CONFIGURATIONS - Compare Different Weighting Strategies

📊 WEIGHTED_STANDARD
   Name: Standard Weighted (current)
   Description: Class imbalance weight + false positive weight + empty instance masking
   Class Weight: True
   FP Weight: True (value: 1.5)
   Empty Masking: True (weight: 0.1)

📊 UNWEIGHTED
   Name: Unweighted (no weights)
   Description: Simple BCE loss without any special weighting
   Class Weight: False
   FP Weight: False (value: N/A)
   Empty Masking: False (weight: N/A)

📊 WEIGHTED_LIGHT
   Name: Light Weighted (reduced)
   Description: Reduced class imbalance and false positive weights
   Class Weight: True
   FP Weight: True (value: 1.2)
   Empty Masking: True (weight: 0.2)

📊 WEIGHTED_CLASS_ONLY
   Name: Class Weight Only
   Description: Only class imbalance weighting, no false positive or empty instance handling
   Class Weight: True
   FP Weight: False (value: N/A)
   Empty Masking: False (weight: N/A)

📊 WEIGHTED_HEAVY
   Name: Heavy Weighted (increased)

In [20]:
# ============================================
# Training Script - Run Through All Configurations + Model Variants
# ============================================

import json
from datetime import datetime
import pandas as pd
import numpy as np

# Define output directory for experiments
outputs_fashion_classification_dir = output_dir  # Use the output_dir from notebook setup

def get_model_class(variant):
    """Get the model class based on variant name"""
    variants = {
        "V1": HierarchicalMultiTaskModel,
        "V2": HierarchicalMultiTaskModelV2,
        "V3": HierarchicalMultiTaskModelV3,
        "V4": HierarchicalMultiTaskModelV4,
    }
    return variants.get(variant, HierarchicalMultiTaskModel)

def get_loss_config(loss_config_name):
    """Get loss configuration from LOSS_CONFIGURATIONS dict"""
    return LOSS_CONFIGURATIONS.get(loss_config_name, LOSS_CONFIGURATIONS["weighted_standard"])

def bce_loss_with_custom_weight(logits, targets, weight=1.0, loss_config=None, device='cpu'):
    """
    Weighted BCE loss with configurable weighting strategy.
    
    Parameters:
    -----------
    logits : torch.Tensor - Model logits
    targets : torch.Tensor - Target labels
    weight : float - Sample-level weight
    loss_config : dict - Loss configuration with different weighting strategies
    device : str - Device to use
    
    Returns:
    --------
    torch.Tensor - Weighted BCE loss
    """
    if loss_config is None:
        loss_config = LOSS_CONFIGURATIONS["weighted_standard"]
    
    # Simple unweighted BCE
    if not loss_config.get('use_class_weight', True) and not loss_config.get('use_false_positive_weight', True) and not loss_config.get('use_empty_instance_masking', True):
        return torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='mean')
    
    # Calculate loss per element
    loss_per_element = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    
    # Initialize combined weight
    combined_weight = torch.ones_like(targets)
    
    # 1. CLASS IMBALANCE WEIGHTING
    if loss_config.get('use_class_weight', True):
        pos_ratio = targets.sum(dim=0) / (targets.shape[0] + 1e-8)
        class_weight = torch.where(
            targets > 0.5,
            torch.full_like(targets, 1.0) / (pos_ratio + 1e-8),  # Weight rare classes higher
            torch.full_like(targets, 1.0) / (1 - pos_ratio + 1e-8)  # Weight common classes lower
        )
        class_weight = class_weight / class_weight.mean()  # Normalize
        combined_weight = combined_weight * class_weight
    
    # 2. FALSE POSITIVE WEIGHTING
    if loss_config.get('use_false_positive_weight', True):
        fp_weight_value = loss_config.get('fp_weight_value', 1.5)
        wrong_prediction = (logits > 0) != (targets > 0.5)
        false_positive_weight = torch.where(
            wrong_prediction,
            torch.full_like(targets, fp_weight_value),
            torch.ones_like(targets)
        )
        combined_weight = combined_weight * false_positive_weight
    
    # 3. EMPTY INSTANCE MASKING
    if loss_config.get('use_empty_instance_masking', True):
        empty_instance_weight = loss_config.get('empty_instance_weight', 0.1)
        # Instances with all zeros get lower weight
        has_labels = targets.sum(dim=1, keepdim=True) > 0
        sample_weight = torch.where(
            has_labels,
            torch.full_like(has_labels, 1.0, dtype=torch.float32),
            torch.full_like(has_labels, empty_instance_weight, dtype=torch.float32)
        )
        combined_weight = combined_weight * sample_weight
    
    # Apply combined weights
    weighted_loss = loss_per_element * combined_weight
    
    # Average per sample then overall
    return weighted_loss.mean()

def convert_to_serializable(obj):
    """Convert numpy/torch types to Python native types for JSON serialization"""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    elif hasattr(obj, 'item') and callable(obj.item):  # numpy scalar or torch tensor
        try:
            return obj.item()
        except:
            return float(obj)
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, float):
        return float(obj)
    elif isinstance(obj, int):
        return int(obj)
    else:
        return obj

def train_configuration(config_name, config, train_loader, val_loader, device='cuda', 
                       epochs=500, early_stop=15):
    """
    Train a single configuration and save outputs in a dedicated folder.
    Folder structure: {outputs_fashion_classification_dir}/{exp_name}/
    Files saved: model.pt, history.json, metrics.json, summary.json, checkpoint.pt
    
    Returns:
        dict with config info, metrics, training history
    """
    print(f"\n{'='*100}")
    print(f"🚀 Training: {config_name.upper()}")
    print(f"{'='*100}")
    
    # Generate experiment name
    exp_name = generate_experiment_name(config_name, config)
    
    # Create experiment folder
    exp_folder = os.path.join(outputs_fashion_classification_dir, exp_name)
    
    # Create file paths inside the experiment folder
    model_path = os.path.join(exp_folder, 'model.pt')
    history_path = os.path.join(exp_folder, 'history.json')
    metrics_path = os.path.join(exp_folder, 'metrics.json')
    summary_path = os.path.join(exp_folder, 'summary.json')
    checkpoint_path = os.path.join(exp_folder, 'checkpoint.pt')
    
    # Skip if already trained
    if os.path.exists(model_path):
        print(f"✓ Already trained: {exp_name}")
        return None
    
    # Create experiment folder
    os.makedirs(exp_folder, exist_ok=True)
    print(f"📁 Saving to: {exp_folder}")
    print(f"📝 Experiment folder: {exp_name}/")
    
    # Get model class based on variant
    ModelClass = get_model_class(config["model_variant"])
    
    # Create model with config
    model = ModelClass(
        embed_dim=FASHIONCLIP_EMBED_DIM,
        num_related=num_related,
        num_categories=num_categories,
        num_main=num_main,
        num_sub=num_sub,
        hidden_dim=config['hidden_dim'],
        dropout=config['dropout'],
        use_attention=True
    )
    
    # Create optimizer based on config
    if config['optimizer'].lower() == 'sgd':
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay'],
            momentum=0.9
        )
    elif config['optimizer'].lower() == 'adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )
    else:  # adamw
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )
    
    # Get loss configuration
    loss_config = get_loss_config(config.get('loss_config', 'weighted_standard'))
    
    print(f"\n📋 Configuration Details:")
    print(f"   Model Variant: {config['model_variant']}")
    print(f"   Optimizer: {config['optimizer'].upper()}")
    print(f"   Learning Rate: {config['learning_rate']}")
    print(f"   Weight Decay: {config['weight_decay']}")
    print(f"   Dropout: {config['dropout']}")
    print(f"   Hidden Dim: {config['hidden_dim']}")
    print(f"   Loss Config: {loss_config['name']}")
    print(f"   Loss Strategy: {loss_config['description']}")
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"   Total Parameters: {num_params:,}")
    
    # Train model
    model, history = train_model(
        model, 
        train_loader, 
        val_loader, 
        optimizer,
        device=device,
        epochs=epochs,
        early_stop=early_stop,
        ckpt_path=checkpoint_path,
        loss_config=loss_config  # Pass loss configuration to train_model
    )
    
    # Save model state
    torch.save(model.state_dict(), model_path)
    print(f"✅ Model saved: {exp_name}/model.pt")
    
    # Evaluate on test set
    print(f"\n📊 Evaluating on test set...")
    test_metrics = evaluate_model(model, test_loader, device=device)
    
    # Convert history and metrics to serializable format
    history = convert_to_serializable(history)
    test_metrics = convert_to_serializable(test_metrics)
    
    # Get test loss
    test_loss = history.get('val_loss', [float('inf')])[-1] if history.get('val_loss') else float('inf')
    
    # Save outputs in experiment folder
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"✅ History saved: {exp_name}/history.json")
    
    # Save metrics in the correct format with test_loss and per_head_metrics
    metrics_to_save = {
        'test_loss': float(test_loss),
        'per_head_metrics': test_metrics
    }
    with open(metrics_path, 'w') as f:
        json.dump(metrics_to_save, f, indent=2)
    print(f"✅ Metrics saved: {exp_name}/metrics.json")
    
    # Save summary
    with open(summary_path, 'w') as f:
        json.dump({
            'config_name': config_name,
            'model_variant': config['model_variant'],
            'num_parameters': num_params,
            'loss_config': loss_config['name'],
            'config': {k: v for k, v in config.items() if k not in ['description', 'model_variant', 'loss_config']},
            'epochs_trained': len(history['train_loss']),
            'final_train_loss': float(history['train_loss'][-1]),
            'final_val_loss': float(history['val_loss'][-1]) if history['val_loss'] else None,
        }, f, indent=2)
    print(f"✅ Summary saved: {exp_name}/summary.json")
    
    # Prepare result summary
    result = {
        'config_name': config_name,
        'exp_name': exp_name,
        'timestamp': datetime.now().isoformat(),
        'model_variant': config['model_variant'],
        'loss_config': loss_config['name'],
        'epochs_trained': len(history['train_loss']),
        'final_train_loss': float(history['train_loss'][-1]),
        'final_val_loss': float(history['val_loss'][-1]) if history['val_loss'] else float('inf'),
        'best_val_loss': min(history['val_loss']) if history['val_loss'] else float('inf'),
        'test_metrics': metrics_to_save,
        'num_parameters': num_params
    }
    
    print(f"\n✅ Training completed!")
    print(f"   Epochs: {result['epochs_trained']}")
    print(f"   Best Val Loss: {result['best_val_loss']:.6f}")
    print(f"   Final Train Loss: {result['final_train_loss']:.6f}")
    if test_metrics and 'related' in test_metrics:
        print(f"   Related F1: {test_metrics['related'].get('f1_micro', 0):.4f}")
    
    return result

# ============================================
# Run all configurations
# ============================================
print("\n" + "="*100)
print("🔥 STARTING MULTI-VARIANT TRAINING GRID WITH DIFFERENT LOSS CONFIGURATIONS")
print("="*100)
print(f"Total configurations: {len(configurations)}")
print(f"Training on: {device}")
print(f"Model Variants: V1, V2, V3, V4")
print(f"Loss Strategies: Weighted (standard/light/heavy) | Unweighted | Class-Only | Empty-Mask-Only")
print(f"Epochs per config: 500 (with early stopping after 15 non-improving epochs)")
print(f"Output directory: {outputs_fashion_classification_dir}")
print("="*100)

# Track all results
all_results = []
start_time = datetime.now()

for i, (config_name, config) in enumerate(configurations.items(), 1):
    print(f"\n[{i}/{len(configurations)}] Starting {config_name}...")
    
    try:
        result = train_configuration(
            config_name,
            config,
            train_loader,
            val_loader,
            device=device,
            epochs=500,
            early_stop=15
        )
        
        if result:
            all_results.append(result)
    except Exception as e:
        print(f"❌ Error training {config_name}: {e}")
        import traceback
        traceback.print_exc()
        

Epoch 82/500 | TrainLoss=0.1400 | ValLoss=0.158819 | BestVal=0.158259


Epoch 83/500 | TrainLoss=0.1406 | ValLoss=0.159730 | BestVal=0.158259


Epoch 84/500 | TrainLoss=0.1398 | ValLoss=0.158274 | BestVal=0.158259


Epoch 85/500 | TrainLoss=0.1390 | ValLoss=0.157698 | BestVal=0.158259


Epoch 86/500 | TrainLoss=0.1386 | ValLoss=0.157385 | BestVal=0.157698


...

Epoch 151/500 | TrainLoss=0.1376 | ValLoss=0.155617 | BestVal=0.155568


Epoch 152/500 | TrainLoss=0.1376 | ValLoss=0.155618 | BestVal=0.155568


Epoch 153/500 | TrainLoss=0.1376 | ValLoss=0.155615 | BestVal=0.155568
⏹️ Early stopping at epoch 153
✅ Model saved: vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei/model.pt

📊 Evaluating on test set...


C:\Users\gorka\AppData\Local\Temp\ipykernel_17932\1908609069.py:77: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



✅ History saved: vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei/history.json
✅ Metrics saved: vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei/metrics.json
✅ Summary saved: vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei/summary.json

✅ Training completed!
   Epochs: 153
   Best Val Loss: 0.155568
   Final Train Loss: 0.137582
   Related F1: 0.1347

[11/11] Starting v1_heavy_weight...

🚀 Training: V1_HEAVY_WEIGHT
✓ Already trained: vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei


In [25]:
# ============================================
# Fashion Classification Model: Metrics Analysis Function
# ============================================

import json
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def analyze_classification_models(parent_folder: str, heads: list[str] = None):
    """
    Generate comprehensive metrics graphs and summaries for classification models.
    
    Args:
        parent_folder: Path to parent folder containing experiment subdirectories
        heads: List of classification heads to analyze (default: ["main", "sub", "category", "related"])
    
    Returns:
        dict: all_experiments with loaded data from all folders
    
    Generates files in 'parent_folder/comparison/' folder:
        - training_curves.html & .png: Train/Val loss across experiments
        - metrics_{head}_comparison.html & .png: Per-head metrics (F1, AUC, Precision, Recall)
        - test_loss_ranking.html & .png: Model ranking by test loss
        - classification_metrics_summary.csv: Summary table of all metrics
    """
    if heads is None:
        heads = ["main", "sub", "category", "related"]
    
    parent_dir = Path(parent_folder)
    outputs_dir = parent_dir / "comparison"
    outputs_dir.mkdir(parents=True, exist_ok=True)
    
    exp_dirs = sorted([d for d in parent_dir.iterdir() if d.is_dir() and d.name != "comparison"])
    
    if not exp_dirs:
        print(f"❌ No experiment folders found in {parent_folder}")
        return {}
    
    print(f"Found {len(exp_dirs)} experiments:")
    for d in exp_dirs:
        print(f"  - {d.name}")
    
    # Load all experiment data
    all_experiments = {}
    for exp_dir in exp_dirs:
        try:
            with open(exp_dir / "history.json") as f:
                history = json.load(f)
            with open(exp_dir / "metrics.json") as f:
                metrics = json.load(f)
            with open(exp_dir / "summary.json") as f:
                summary = json.load(f)
            all_experiments[exp_dir.name] = {
                "history": history,
                "metrics": metrics,
                "summary": summary
            }
        except Exception as e:
            print(f"  ⚠️  Error loading {exp_dir.name}: {e}")
    
    print(f"\n✅ Loaded {len(all_experiments)} experiments successfully\n")
    
    # ===== Plot training curves =====
    print("📊 Generating training curves...")
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Training Loss", "Validation Loss"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    for exp_name, data in all_experiments.items():
        hist = data["history"]
        epochs = list(range(1, len(hist["train_loss"]) + 1))
        
        fig.add_trace(
            go.Scatter(
                x=epochs,
                y=hist["train_loss"],
                mode="lines",
                name=f"{exp_name} (train)",
                line=dict(width=2)
            ),
            row=1, col=1
        )
        
        if "val_loss" in hist:
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=hist["val_loss"],
                    mode="lines",
                    name=f"{exp_name} (val)",
                    line=dict(width=2, dash="dash")
                ),
                row=1, col=2
            )
    
    fig.update_xaxes(title_text="Epoch", row=1, col=1)
    fig.update_xaxes(title_text="Epoch", row=1, col=2)
    fig.update_yaxes(title_text="Loss", row=1, col=1)
    fig.update_yaxes(title_text="Loss", row=1, col=2)
    fig.update_layout(height=500, width=1400, title_text="Training Curves Across Experiments")
    fig.write_html(str(outputs_dir / "training_curves.html"))
    fig.write_image(str(outputs_dir / "training_curves.png"), width=1400, height=500)
    fig.write_image(str(outputs_dir / "training_curves.jpg"), width=1400, height=500)
    fig.show()
    print("  ✅ Saved: training_curves.html, training_curves.png, training_curves.jpg")
    
    # ===== Per-head metrics comparison =====
    print("\n📊 Generating per-head metrics...")
    for head in heads:
        fig = go.Figure()
        
        for exp_name, data in all_experiments.items():
            metrics = data["metrics"].get("per_head_metrics", {}).get(head, {})
            if not metrics:
                continue
                
            f1_micro = metrics.get("f1_micro", 0)
            f1_macro = metrics.get("f1_macro", 0)
            f1_weighted = metrics.get("f1_weighted", 0)
            auc_mean = metrics.get("auc_mean", 0)
            precision = metrics.get("precision_micro", 0)
            recall = metrics.get("recall_micro", 0)
            hamming_loss = metrics.get("hamming_loss", 0)
            subset_acc = metrics.get("subset_acc", 0)
            
            fig.add_trace(go.Bar(
                name=exp_name,
                x=["F1-Micro", "F1-Macro", "F1-Weighted", "AUC", "Precision", "Recall", "Hamming Loss", "Subset Acc"],
                y=[f1_micro, f1_macro, f1_weighted, auc_mean, precision, recall, hamming_loss, subset_acc],
                text=[f"{v:.3f}" for v in [f1_micro, f1_macro, f1_weighted, auc_mean, precision, recall, hamming_loss, subset_acc]],
                textposition="auto"
            ))
        
        fig.update_layout(
            title=f"{head.upper()} Classification Metrics",
            barmode="group",
            height=500,
            width=1200,
            yaxis_title="Score",
            yaxis=dict(range=[0, 1.05])
        )
        fig.write_html(str(outputs_dir / f"metrics_{head}_comparison.html"))
        fig.write_image(str(outputs_dir / f"metrics_{head}_comparison.png"), width=1200, height=500)
        fig.write_image(str(outputs_dir / f"metrics_{head}_comparison.jpg"), width=1200, height=500)
        fig.show()
        print(f"  ✅ Saved: metrics_{head}_comparison.html, .png, .jpg")
    
    # ===== Create comprehensive summary table =====
    print("\n📊 Creating summary table...")
    summary_rows = []
    for exp_name, data in all_experiments.items():
        metrics = data["metrics"]
        per_head = metrics.get("per_head_metrics", {})
        
        row = {"experiment": exp_name}
        row["test_loss"] = metrics.get("test_loss", 0)
        
        for head in heads:
            head_metrics = per_head.get(head, {})
            row[f"{head}_f1_micro"] = head_metrics.get("f1_micro", 0)
            row[f"{head}_f1_macro"] = head_metrics.get("f1_macro", 0)
            row[f"{head}_f1_weighted"] = head_metrics.get("f1_weighted", 0)
            row[f"{head}_auc"] = head_metrics.get("auc_mean", 0)
            row[f"{head}_precision"] = head_metrics.get("precision_micro", 0)
            row[f"{head}_recall"] = head_metrics.get("recall_micro", 0)
            row[f"{head}_hamming_loss"] = head_metrics.get("hamming_loss", 0)
            row[f"{head}_subset_acc"] = head_metrics.get("subset_acc", 0)
        
        summary_rows.append(row)
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.round(4)
    
    print("\n" + "="*160)
    print("COMPREHENSIVE METRICS SUMMARY")
    print("="*160)
    print(summary_df.to_string(index=False))
    print("="*160)
    
    summary_df.to_csv(str(outputs_dir / "classification_metrics_summary.csv"), index=False)
    print("  ✅ Saved: classification_metrics_summary.csv")
    
    # ===== Test Loss Comparison & Model Ranking =====
    print("\n📊 Generating test loss ranking...")
    test_loss_data = [(exp_name, all_experiments[exp_name]["metrics"].get("test_loss", float('inf'))) 
                      for exp_name in all_experiments.keys()]
    test_loss_data.sort(key=lambda x: x[1])
    
    fig_loss = go.Figure(go.Bar(
        y=[name for name, _ in test_loss_data],
        x=[loss for _, loss in test_loss_data],
        orientation='h',
        text=[f"{loss:.4f}" for _, loss in test_loss_data],
        textposition='auto',
        marker=dict(
            color=[loss for _, loss in test_loss_data],
            colorscale='RdYlGn_r',
            showscale=True
        )
    ))
    
    fig_loss.update_layout(
        title="Test Loss: Model Ranking",
        xaxis_title="Test Loss (Lower is Better)",
        height=400,
        width=900,
        showlegend=False
    )
    fig_loss.write_html(str(outputs_dir / "test_loss_ranking.html"))
    fig_loss.write_image(str(outputs_dir / "test_loss_ranking.png"), width=900, height=400)
    fig_loss.write_image(str(outputs_dir / "test_loss_ranking.jpg"), width=900, height=400)
    fig_loss.show()
    print("  ✅ Saved: test_loss_ranking.html, .png, .jpg")
    
    # ===== Average metrics summary =====
    print("\n" + "="*100)
    print("AVERAGE METRICS ACROSS ALL MODELS")
    print("="*100)
    for head in heads:
        avg_f1_micro = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("f1_micro", 0) 
                          for e in all_experiments.keys()])
        avg_f1_macro = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("f1_macro", 0) 
                          for e in all_experiments.keys()])
        avg_f1_weighted = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("f1_weighted", 0) 
                          for e in all_experiments.keys()])
        avg_auc = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("auc_mean", 0) 
                           for e in all_experiments.keys()])
        avg_precision = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("precision_micro", 0) 
                           for e in all_experiments.keys()])
        avg_recall = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("recall_micro", 0) 
                           for e in all_experiments.keys()])
        avg_hamming_loss = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("hamming_loss", 0) 
                           for e in all_experiments.keys()])
        avg_subset_acc = np.mean([all_experiments[e]["metrics"].get("per_head_metrics", {}).get(head, {}).get("subset_acc", 0) 
                           for e in all_experiments.keys()])
        
        print(f"\n{head.upper()}")
        print(f"  F1-Micro: {avg_f1_micro:.4f} | F1-Macro: {avg_f1_macro:.4f} | F1-Weighted: {avg_f1_weighted:.4f}")
        print(f"  AUC: {avg_auc:.4f} | Precision: {avg_precision:.4f} | Recall: {avg_recall:.4f}")
        print(f"  Hamming Loss: {avg_hamming_loss:.6f} | Subset Accuracy: {avg_subset_acc:.4f}")
    
    avg_test_loss = np.mean([all_experiments[e]["metrics"].get("test_loss", 0) for e in all_experiments.keys()])
    print(f"\n{'='*100}")
    print(f"Average Test Loss: {avg_test_loss:.4f}")
    print("="*100)
    
    return all_experiments

# Example usage:


all_experiments = analyze_classification_models("outputs_fashion_classification")


Found 10 experiments:
  - vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_lossunw
  - vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei
  - vV1_hd512_dr20_adamw_lr5e-4_wd1e-4_losswei
  - vV2_hd512_dr25_adamw_lr5e-4_wd1e-4_losswei
  - vV2_hd768_dr30_adamw_lr5e-4_wd1e-4_losswei
  - vV3_hd512_dr20_adamw_lr1e-3_wd1e-4_losswei
  - vV3_hd768_dr25_adamw_lr1e-3_wd5e-5_losswei
  - vV4_hd512_dr25_adamw_lr1e-3_wd1e-4_lossunw
  - vV4_hd512_dr25_adamw_lr1e-3_wd1e-4_losswei
  - vV4_hd768_dr30_adamw_lr1e-3_wd5e-5_losswei

✅ Loaded 10 experiments successfully

📊 Generating training curves...


  ✅ Saved: training_curves.html, training_curves.png, training_curves.jpg

📊 Generating per-head metrics...


  ✅ Saved: metrics_main_comparison.html, .png, .jpg


  ✅ Saved: metrics_sub_comparison.html, .png, .jpg


  ✅ Saved: metrics_category_comparison.html, .png, .jpg


  ✅ Saved: metrics_related_comparison.html, .png, .jpg

📊 Creating summary table...

COMPREHENSIVE METRICS SUMMARY
                                experiment  test_loss  main_f1_micro  main_f1_macro  main_f1_weighted  main_auc  main_precision  main_recall  main_hamming_loss  main_subset_acc  sub_f1_micro  sub_f1_macro  sub_f1_weighted  sub_auc  sub_precision  sub_recall  sub_hamming_loss  sub_subset_acc  category_f1_micro  category_f1_macro  category_f1_weighted  category_auc  category_precision  category_recall  category_hamming_loss  category_subset_acc  related_f1_micro  related_f1_macro  related_f1_weighted  related_auc  related_precision  related_recall  related_hamming_loss  related_subset_acc
vV1_hd512_dr20_adamw_lr1e-3_wd1e-4_lossunw     0.0346         0.9695         0.9563            0.9692    0.9968          0.9749       0.9642             0.0058           0.9597        0.8350        0.4123           0.8181   0.9854         0.8786      0.7955            0.0039          0.7406

  ✅ Saved: test_loss_ranking.html, .png, .jpg

AVERAGE METRICS ACROSS ALL MODELS

MAIN
  F1-Micro: 0.9612 | F1-Macro: 0.9437 | F1-Weighted: 0.9619
  AUC: 0.9963 | Precision: 0.9454 | Recall: 0.9777
  Hamming Loss: 0.007620 | Subset Accuracy: 0.9446

SUB
  F1-Micro: 0.7110 | F1-Macro: 0.4441 | F1-Weighted: 0.7614
  AUC: 0.9874 | Precision: 0.5947 | Recall: 0.9336
  Hamming Loss: 0.009808 | Subset Accuracy: 0.4461

CATEGORY
  F1-Micro: 0.4967 | F1-Macro: 0.1956 | F1-Weighted: 0.6208
  AUC: 0.9950 | Precision: 0.3795 | Recall: 0.9199
  Hamming Loss: 0.009799 | Subset Accuracy: 0.5215

RELATED
  F1-Micro: 0.1404 | F1-Macro: 0.1341 | F1-Weighted: 0.1774
  AUC: 0.9875 | Precision: 0.1667 | Recall: 0.7942
  Hamming Loss: 0.028316 | Subset Accuracy: 0.5616

Average Test Loss: 0.1330


In [ ]:
## 2.5. Generate and Save Attribute Embeddings (Independent Cell)

import os
import sys
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# ============ SETUP (INDEPENDENT) ============
print("="*70)
print("INDEPENDENT EMBEDDING GENERATION")
print("="*70)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n📱 Device: {device}")

# Paths (independent setup)
BASE_PATH = Path("c:/TFM/APP")
DATA_FOLDER = BASE_PATH / "ml_pipeline/data"
MODELS_PATH = BASE_PATH / "ml_pipeline/models/base"

TRAIN_ITEMS_PATH = DATA_FOLDER / "train_items.csv"
VAL_ITEMS_PATH = DATA_FOLDER / "val_items.csv"
TEST_ITEMS_PATH = DATA_FOLDER / "test_items.csv"

print(f"\n📂 Paths configured:")
print(f"   Data folder: {DATA_FOLDER}")
print(f"   Models folder: {MODELS_PATH}")

# ============ DEFINE ATTRIBUTE ENCODER ============
class AttributeEncoder(nn.Module):
    """Encodes categorical attributes into normalized embeddings."""
    def __init__(self, input_dim, output_dim=256, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.encoder(x)
        return F.normalize(x, p=2, dim=-1)

# ============ HELPER FUNCTIONS ============
def parse_list_value(v):
    """Parse a value that might be string, list, or empty."""
    if isinstance(v, (list, np.ndarray)):
        return list(v)
    if isinstance(v, str):
        if v.strip() == "" or v.strip() == "[]":
            return []
        try:
            return ast.literal_eval(v)
        except Exception:
            return []
    return []

def encode_list_column(df, col, max_index):
    """Encode list column to binary matrix."""
    binary_matrix = np.zeros((len(df), max_index), dtype=np.float32)
    for i, values in enumerate(df[col]):
        parsed_values = parse_list_value(values)
        for v in parsed_values:
            try:
                v = int(v)
                if 0 <= v < max_index:
                    binary_matrix[i, v] = 1.0
            except (ValueError, TypeError):
                pass
    return binary_matrix

# ============ LOAD CATEGORY DIMENSIONS FROM CSV ============
print("\n📊 Loading category mapping files...")
idx2related = pd.read_csv(MODELS_PATH / "idx2related.csv")
idx2category = pd.read_csv(MODELS_PATH / "idx2category.csv")
idx2main = pd.read_csv(MODELS_PATH / "idx2main.csv")
idx2sub = pd.read_csv(MODELS_PATH / "idx2sub.csv")

# Validate structure
required_files = {
    'idx2related': idx2related,
    'idx2category': idx2category,
    'idx2main': idx2main,
    'idx2sub': idx2sub
}

for file_name, df_obj in required_files.items():
    if 'index' not in df_obj.columns or 'name' not in df_obj.columns:
        print(f"❌ {file_name}.csv missing required columns: expected 'index' and 'name'")
        raise ValueError(f"Invalid structure in {file_name}.csv")

max_related = len(idx2related)
max_category = len(idx2category)
max_main = len(idx2main)
max_sub = len(idx2sub)

print(f"✓ Category dimensions loaded:")
print(f"  - Related: {max_related} categories")
print(f"  - Category: {max_category} categories")
print(f"  - Main: {max_main} categories")
print(f"  - Sub: {max_sub} categories")

# ============ LOAD ITEMS INDEPENDENTLY ============
print("\n📂 Loading items data...")
df_train_items = pd.read_csv(TRAIN_ITEMS_PATH)
df_val_items = pd.read_csv(VAL_ITEMS_PATH)
df_test_items = pd.read_csv(TEST_ITEMS_PATH)

print(f"✓ Train items: {len(df_train_items)} rows")
print(f"✓ Val items: {len(df_val_items)} rows")
print(f"✓ Test items: {len(df_test_items)} rows")

# Validate required columns exist
required_cols = ["related_indices", "category_indices", "main_category_indices", "sub_category_indices"]
for dataset_name, df in [("Train", df_train_items), ("Val", df_val_items), ("Test", df_test_items)]:
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ {dataset_name} missing columns: {missing_cols}")
        raise ValueError(f"Missing required columns in {dataset_name} items")

print(f"✓ All required columns present")

# ============ GENERATE EMBEDDINGS FOR TRAIN DATA ============
print("\n🔧 Generating training embeddings...")
related_bin_train = encode_list_column(df_train_items, "related_indices", max_related)
category_bin_train = encode_list_column(df_train_items, "category_indices", max_category)
main_bin_train = encode_list_column(df_train_items, "main_category_indices", max_main)
sub_bin_train = encode_list_column(df_train_items, "sub_category_indices", max_sub)

X_attr_train = np.concatenate([related_bin_train, category_bin_train, main_bin_train, sub_bin_train], axis=1)
input_dim = X_attr_train.shape[1]

print(f"✓ Train attribute matrix: {X_attr_train.shape}")

# Create and train encoder on train data
attribute_encoder = AttributeEncoder(input_dim=input_dim).to(device)
X_tensor_train = torch.tensor(X_attr_train, dtype=torch.float32).to(device)

attribute_encoder.eval()
with torch.no_grad():
    embeddings_train = attribute_encoder(X_tensor_train).cpu().numpy()

df_train_items["Xa"] = list(embeddings_train)
print(f"✓ Train embeddings generated: {embeddings_train.shape}")
print(f"  Embedding dimension: {embeddings_train.shape[1]}")

# ============ GENERATE EMBEDDINGS FOR VAL DATA ============
print("\n🔧 Generating validation embeddings...")
related_bin_val = encode_list_column(df_val_items, "related_indices", max_related)
category_bin_val = encode_list_column(df_val_items, "category_indices", max_category)
main_bin_val = encode_list_column(df_val_items, "main_category_indices", max_main)
sub_bin_val = encode_list_column(df_val_items, "sub_category_indices", max_sub)

X_attr_val = np.concatenate([related_bin_val, category_bin_val, main_bin_val, sub_bin_val], axis=1)
X_tensor_val = torch.tensor(X_attr_val, dtype=torch.float32).to(device)

with torch.no_grad():
    embeddings_val = attribute_encoder(X_tensor_val).cpu().numpy()

df_val_items["Xa"] = list(embeddings_val)
print(f"✓ Val embeddings generated: {embeddings_val.shape}")

# ============ GENERATE EMBEDDINGS FOR TEST DATA ============
print("\n🔧 Generating test embeddings...")
related_bin_test = encode_list_column(df_test_items, "related_indices", max_related)
category_bin_test = encode_list_column(df_test_items, "category_indices", max_category)
main_bin_test = encode_list_column(df_test_items, "main_category_indices", max_main)
sub_bin_test = encode_list_column(df_test_items, "sub_category_indices", max_sub)

X_attr_test = np.concatenate([related_bin_test, category_bin_test, main_bin_test, sub_bin_test], axis=1)
X_tensor_test = torch.tensor(X_attr_test, dtype=torch.float32).to(device)

with torch.no_grad():
    embeddings_test = attribute_encoder(X_tensor_test).cpu().numpy()

df_test_items["Xa"] = list(embeddings_test)
print(f"✓ Test embeddings generated: {embeddings_test.shape}")

# ============ SAVE ALL ENRICHED DATASETS ============
print("\n💾 Saving enriched datasets with embeddings...")

try:
    df_train_items.to_csv(TRAIN_ITEMS_PATH, index=False)
    print(f"✓ Saved train items: {TRAIN_ITEMS_PATH}")
    print(f"  Rows: {len(df_train_items)}, Columns: {len(df_train_items.columns)}")
except Exception as e:
    print(f"❌ Error saving train items: {e}")
    raise

try:
    df_val_items.to_csv(VAL_ITEMS_PATH, index=False)
    print(f"✓ Saved val items: {VAL_ITEMS_PATH}")
    print(f"  Rows: {len(df_val_items)}, Columns: {len(df_val_items.columns)}")
except Exception as e:
    print(f"❌ Error saving val items: {e}")
    raise

try:
    df_test_items.to_csv(TEST_ITEMS_PATH, index=False)
    print(f"✓ Saved test items: {TEST_ITEMS_PATH}")
    print(f"  Rows: {len(df_test_items)}, Columns: {len(df_test_items.columns)}")
except Exception as e:
    print(f"❌ Error saving test items: {e}")
    raise


# ============ SAVE ATTRIBUTE ENCODER MODEL ============
print("\n💾 Saving attribute encoder model...")

encoder_path = MODELS_PATH / "attribute_encoder.pt"
try:
    torch.save(attribute_encoder.state_dict(), encoder_path)
    print(f"✓ Saved attribute encoder: {encoder_path}")
    
    # Verify file size
    file_size_mb = encoder_path.stat().st_size / (1024 * 1024)
    print(f"  File size: {file_size_mb:.2f} MB")
    print(f"  Model parameters: {sum(p.numel() for p in attribute_encoder.parameters()):,}")
except Exception as e:
    print(f"❌ Error saving encoder: {e}")
    raise

# ============ VERIFICATION ============
print("\n✅ Verification:")
print(f"  Train: {len(df_train_items)} items with Xa column ✓")
print(f"  Val: {len(df_val_items)} items with Xa column ✓")
print(f"  Test: {len(df_test_items)} items with Xa column ✓")
print(f"  Encoder model saved ✓")

print(f"\n" + "="*70)
print("✅ INDEPENDENT EMBEDDING GENERATION COMPLETE")
print("="*70)
print(f"\nThis cell is fully independent and can run without other cells.")
print(f"All data loaded, processed, and saved within this cell.")
